In [5]:
#!/usr/bin/env python3
"""
Receive and decode BPSK with:
- Matched Filtering (RRC)
- Timing Recovery (Gardner loop)
- Carrier Recovery (Costas loop)
- Preamble-based Frame Sync

via UHD on a USRP B210.

NOTE: This is a conceptual demonstration; it's not rigorously tested or tuned.
"""

import uhd
import time
import numpy as np
import scipy.signal as signal

#########################
# Configurable Settings #
#########################
RX_RATE   = 500e3   # Must match TX (symbol_rate*sps)
FREQ      = 2.45e9  # Must match TX
FREQ = uhd.libpyuhd.types.tune_request(FREQ)
GAIN      = 30.0    # RX gain
SPS       = 4       # Same as TX
ALPHA     = 0.35    # RRC roll-off
NUM_TAPS  = 101     # RRC filter length
PREAMBLE  = [1,1,1,0,0,0,1,0,0,0,1,0,1]  # Barker-ish pattern from TX
#######################################################

def rrc_filter(num_taps, alpha, sps):
    """
    Generate RRC filter. Same as TX side.
    """
    t = np.arange(-num_taps//2, num_taps//2 + 1, dtype=np.float64) / sps
    h = []
    for ti in t:
        if ti == 0.0:
            val = 1.0 - alpha + 4*alpha/np.pi
        else:
            numerator   = (np.sin(np.pi*ti*(1-alpha)) +
                           4*alpha*ti*np.cos(np.pi*ti*(1+alpha)))
            denominator = np.pi*ti * (1 - (4*alpha*ti)**2)
            if abs(denominator) < 1e-10:
                val = 1.0 - alpha + 4*alpha/np.pi
            else:
                val = numerator / denominator
        h.append(val)
    h = np.array(h, dtype=np.float64)
    # Normalize
    h /= np.sqrt(np.sum(h*h))
    return h.astype(np.float32)

def gardner_timing_recovery(samples, sps, loop_gain=0.01, damping=1.0):
    """
    Simple Gardner timing recovery (conceptual).
    - samples: matched-filtered baseband signal, oversampled by 'sps'
    - sps: nominal samples per symbol
    - loop_gain, damping: loop parameters (very sensitive!)

    Returns an array of symbol-spaced samples.
    """
    # We'll store the recovered samples here
    out = []
    # mu: fractional offset in [0, sps)
    mu = 0.0

    # Loop constants (very approximate)
    # For robust design, these need to be carefully chosen based on loop bandwidth.
    alpha = loop_gain
    beta = alpha**2 / damping

    for i in range(len(samples) - sps):  # avoid going out of bounds
        # Current sample at fractional offset
        idx = int(i + mu)
        current = samples[min(idx,len(samples)-1)]
        # Sample 1 symbol ago
        idx_prev = int(i + mu - sps)
        if idx_prev < 0:
            continue
        previous = samples[min(idx_prev,len(samples)-1)]

        # Gardner error
        # e = (y(n - T/2) * (x(n) - x(n - T)) ) in a typical formula.
        # But let's do a simplified version:
        e = (np.real(previous) * np.sign(np.real(current))) + \
            (np.imag(previous) * np.sign(np.imag(current)))

        # Update
        mu += beta * e
        # integer step every iteration
        mu += 1.0

        out.append(current)

    return np.array(out, dtype=np.complex64)

def costas_loop_bpsk(samples, loop_gain=0.001, damping=1.0):
    """
    Simple Costas loop for BPSK (conceptual).
    Returns frequency/phase-corrected samples.
    """
    phase = 0.0
    freq  = 0.0

    alpha = loop_gain
    beta  = alpha**2 / damping

    out = np.zeros_like(samples, dtype=np.complex64)
    for i, x in enumerate(samples):
        # Rotate by -phase
        rot = x * np.exp(-1j*phase)
        # Decision device for BPSK: sign of real
        # We only care about the real axis, but let's keep it complex
        decision = np.sign(np.real(rot)) + 0j

        # Phase error = imag(decision * conj(rot))
        # For BPSK, that might be real(decision)*imag(rot) - imag(decision)*real(rot),
        # but let's keep it simpler:
        error = np.imag(decision * np.conjugate(rot))

        # Update freq & phase
        freq  += beta  * error
        phase += freq  + alpha * error

        out[i] = rot

    return out

def bits_to_string(bit_list):
    """
    Convert list/array of bits to ASCII string (8-bit, MSB first).
    """
    chars = []
    for i in range(0, len(bit_list), 8):
        byte_bits = bit_list[i:i+8]
        if len(byte_bits) < 8:
            break
        val = 0
        for b in byte_bits:
            val = (val << 1) | b
        chars.append(chr(val))
    return "".join(chars)

def main():
    #------------------------------------
    # 1. Setup USRP
    #------------------------------------
    usrp = uhd.usrp.MultiUSRP()
    usrp.set_rx_rate(RX_RATE)
    usrp.set_rx_freq(FREQ, 0)
    usrp.set_rx_gain(GAIN, 0)
    usrp.set_rx_agc(False, 0)

    #------------------------------------
    # 2. Create RX Streamer
    #------------------------------------
    stream_args = uhd.usrp.StreamArgs("fc32", "sc16")
    stream_args.channels = [0]
    rx_streamer = usrp.get_rx_stream(stream_args)

    # We'll try to capture enough samples to (hopefully) catch
    # our repeated bursts. Adjust for your real scenario.
    num_requested_samples = 200000
    recv_buffer = np.zeros(num_requested_samples, dtype=np.complex64)

    #------------------------------------
    # 3. Start streaming
    #------------------------------------
    md = uhd.types.RXMetadata()

    # Start Stream
    stream_cmd = uhd.types.StreamCMD(uhd.types.StreamMode.start_cont)
    stream_cmd.stream_now = True
    rx_streamer.issue_stream_cmd(stream_cmd)

    time.sleep(0.1)  # let samples accumulate

    #------------------------------------
    # 4. Receive
    #------------------------------------
    print("[RX] Receiving samples...")
    total_recvd = 0
    while total_recvd < num_requested_samples:
        samps = rx_streamer.recv(recv_buffer[total_recvd:], md, timeout=2.0)
        if samps == 0:
            print("[RX] Timeout or no samples received!")
            break
        total_recvd += samps

    rx_streamer.issue_stream_cmd(
        uhd.types.StreamCMD(uhd.types.StreamMode.stop_cont)
    )
    print(f"[RX] Received {total_recvd} samples.")

    #------------------------------------
    # 5. Matched filter (RRC)
    #------------------------------------
    rrc = rrc_filter(NUM_TAPS, ALPHA, SPS)
    filtered = signal.lfilter(rrc, 1.0, recv_buffer[:total_recvd])

    #------------------------------------
    # 6. Timing Recovery (Gardner)
    #------------------------------------
    symbol_aligned = gardner_timing_recovery(filtered, SPS, loop_gain=0.005, damping=1.0)

    #------------------------------------
    # 7. Carrier Recovery (Costas for BPSK)
    #------------------------------------
    freq_corrected = costas_loop_bpsk(symbol_aligned, loop_gain=0.001, damping=1.0)

    #------------------------------------
    # 8. Frame Sync (Preamble Correlation)
    #------------------------------------
    # We'll do a simple correlation with the known preamble (BPSK mapped).
    # Preamble bits: PREAMBLE
    preamble_bits = np.array(PREAMBLE, dtype=np.uint8)
    preamble_syms = np.where(preamble_bits == 0, 1.0, -1.0).astype(np.complex64)

    # We'll search the real part only (since BPSK is on real axis).
    corr = np.correlate(np.real(freq_corrected), np.real(preamble_syms), mode='valid')
    start_idx = np.argmax(np.abs(corr))  # approximate location of preamble

    print(f"[RX] Frame sync: best correlation at sample index {start_idx}")

    #------------------------------------
    # 9. Hard Decision Demod
    #------------------------------------
    # We'll slice out some chunk after the preamble.
    # The length of preamble is len(PREAMBLE).
    # Each bit is 1 symbol => we expect preamble to occupy len(PREAMBLE) symbols
    # Then our payload is unknown length, but let's guess we have ~ (rest) symbols.
    # For a real system, you'd structure a length field or end-of-frame marker.
    demod_syms = freq_corrected[start_idx : ]
    # Decision: real >= 0 => bit=0, else bit=1
    # We'll ignore imag for BPSK
    rx_bits = (np.real(demod_syms) < 0).astype(np.uint8)

    # We know the preamble length in bits
    preamble_len = len(PREAMBLE)
    # So the bits after that are the payload:
    payload_bits = rx_bits[preamble_len:]  # skip the preamble bits

    #------------------------------------
    # 10. Convert bits -> ASCII
    #------------------------------------
    decoded_msg = bits_to_string(payload_bits)
    print("[RX] Decoded message (ASCII):")
    print(decoded_msg)

    print("[RX] Done.")

if __name__ == "__main__":
    main()


[RX] Receiving samples...
[RX] Received 200000 samples.
[RX] Frame sync: best correlation at sample index 37749
[RX] Decoded message (ASCII):
ð À  ð   0=Àáÿû7Üwã¸=< ` 3À88ø ÿ ûÈ>  !û=àÿýçÁÿþ~ó?üqÿÇ?ÿÿïñóïÀ íÿð|?þ¸0øÿÿðÌçî ñÿÀáÇ= ùÿ¸ÿóÿÿÿñÏÿùàðÿþã0?à@?00` À3ðÿÿÿÿÿÃßð3ÿþ|ÿÿÿÝÿÇøàÿ¸àñülþà¿Þgçÿÿ÷àÁùãÃÆ?|ãsÿÆs |üÀ 	>øyÿàüïùÿùÿÿïÃø8ä  BØpp1; ÇöÑÿÆ0	Ø68yÿÇÿÿ~ ñÜà `ï<p x\÷    xÇî<ÿÿ¸ð>àp<v9þÿÆf  ?8üáøÀÈÀb#Î 0ñp ý ??àà{ýÀn?þÿÜÍãÁà##|lb@óç?1ÿÏÀ Ç0<  <ðî~?ÿ~¿þ?ÿþøÇÎp8! À ð 0äóÿñþïÁÏ89ãà|pÍÇ3ùûóâ x÷ ¸ Äfð  À<|?À0~~ïûÇþ>{¹~pðà  @ 0  !øøØ  Î<	ýó¿ÿÿ¾Æ 0Àÿ 8Dÿü0@ÿ3áÄ@ 0  ð1Ïæ#ÿð ;;  ùàÎwÿð÷ø8ïÿ}ïÏÿóÿÿ8¸~xÿ0Àb ãOüàÇ þ?~xð$>ÿøßáÿËðà }ßøß>}ß»ó0ø¿ÏÇ` 8ð Á=0	   xøó   0@ >ÁßïÿÿçýÌ?ÿûçÿÿÿÆóÿüçÏþ~3`> ÿÿÿÿ`ÿþàÀ `ægø{Ø;ÿþ~?ÇþìÍáß?ÿïü8ÿó3ÿxÿðü 0>!ýäø0àð    @  B 9Ïÿÿ¼Þ<|Ç